# Exploratory Data Analysis (EDA) Project

**Goal:** Analyze a tabular dataset to uncover patterns and trends.

This notebook is written for someone starting from zero. Run the cells **from top to bottom**.

### Assignment requirements covered
- Statistical summaries and visualizations
- Correlations and key influencing factors
- Structured insights/report


## Before you start

**EDA** means *Exploratory Data Analysis*. In simple words, we inspect a dataset before building a machine learning model.

Think of it like this:

**Load → Inspect → Clean → Visualize → Compare → Find patterns → Explain findings**

The ready-to-run example uses the **Titanic dataset**. Later, you can replace it with your own CSV file.


In [ ]:
# STEP 0: Install/import the libraries
# In Google Colab, run this cell first.

!pip -q install pandas numpy matplotlib seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")


# Step 1 — Data Loading & Initial Inspection

### What are we checking?
- **Rows:** individual records
- **Columns:** variables/features
- **Data type:** number, text/category, True/False, etc.
- **Shape:** `(rows, columns)`

The first few rows help us understand what the data actually looks like.


In [ ]:
# STEP 1: Load the dataset

# Ready-to-run example:
df = sns.load_dataset("titanic")

# If you want to use your own CSV later, use:
# df = pd.read_csv("data/your_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()


### Simple interpretation

If the shape is `(891, 15)`, that means:
- 891 rows/records
- 15 columns/variables

A **numerical column** contains numbers, such as age or fare.

A **categorical column** contains groups/labels, such as sex or class.


# Step 2 — Summary Statistics

### Important statistical terms

- **Mean:** average value.
- **Median:** middle value after sorting. It is often more resistant to extreme values than the mean.
- **Standard deviation:** how spread out values are around the mean.
- **Minimum/Maximum:** smallest/largest value.
- **Count:** number of non-missing observations.
- **25%, 50%, 75%:** quartiles that help describe the distribution.

For categorical variables, we look at counts and the most common category.


In [ ]:
# STEP 2: Separate numerical and categorical columns

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

print("\nNumerical descriptive statistics:")
display(df[numeric_cols].describe().T)

print("\nCategorical descriptive statistics:")
if categorical_cols:
    display(df[categorical_cols].describe().T)
else:
    print("No categorical columns found.")


# Step 3 — Missing Data & Duplicates

### Missing data
A missing value means information was not recorded.

We first **identify** missing values before deciding what to do.

### Duplicates
A duplicate row is a repeated record. We count duplicates and remove exact duplicate rows when appropriate.

### Important
There is no single correct way to handle every missing value. The correct choice depends on what the column means and why values are missing.


In [ ]:
# STEP 3: Check missing values and duplicates

missing_count = df.isnull().sum()
missing_percent = (df.isnull().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent
}).sort_values("Missing Count", ascending=False)

print("Missing-value report:")
display(missing_table[missing_table["Missing Count"] > 0])

print("Number of duplicate rows:", df.duplicated().sum())


In [ ]:
# Create a cleaned copy so the original data remains unchanged.

df_clean = df.drop_duplicates().copy()

# Fill missing numerical values with the median.
# Median is a common simple choice when outliers may exist.
for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Fill missing categorical values with the most common category (mode).
for col in categorical_cols:
    if df_clean[col].isnull().any():
        mode = df_clean[col].mode(dropna=True)
        if len(mode) > 0:
            df_clean[col] = df_clean[col].fillna(mode.iloc[0])
        else:
            df_clean[col] = df_clean[col].fillna("Unknown")

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Total missing values after cleaning:", int(df_clean.isnull().sum().sum()))


# Step 4 — Univariate Analysis

**Univariate** means studying one variable at a time.

### Charts we will use
- **Histogram:** shows how a numerical variable is distributed.
- **Box plot:** helps identify spread and possible outliers.
- **Bar/count plot:** shows how often categories occur.

### Outlier
An outlier is a value that is unusually far from the rest of the observations. An outlier is not automatically an error; it may be a real observation.


In [ ]:
# STEP 4A: Histograms for numerical columns

for col in numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df_clean[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 4B: Box plots for numerical columns

for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df_clean[col])
    plt.title(f"Box Plot of {col}")
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 4C: Bar/count charts for categorical columns
# We skip columns with too many unique categories to keep the charts readable.

for col in categorical_cols:
    if df_clean[col].nunique() <= 20:
        plt.figure(figsize=(9, 5))
        order = df_clean[col].value_counts().index
        sns.countplot(data=df_clean, x=col, order=order)
        plt.title(f"Category Counts: {col}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


# Step 5 — Bivariate & Multivariate Analysis

**Bivariate** analysis compares two variables.

**Multivariate** analysis examines several variables together.

### Correlation
Correlation measures how strongly two numerical variables move together.

- Close to **+1** → strong positive relationship
- Close to **-1** → strong negative relationship
- Close to **0** → weak/no linear relationship

**Important:** correlation does not prove causation.


In [ ]:
# STEP 5A: Correlation matrix and heatmap

corr = df_clean[numeric_cols].corr()

print("Correlation matrix:")
display(corr.round(2))

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


### How to read a heatmap

Look for cells with values close to:
- **+1:** variables increase together
- **-1:** one tends to increase while the other decreases
- **0:** little linear relationship

Do not call a relationship "causal" just because the correlation is high.


In [ ]:
# STEP 5B: Pair plot
# This compares every numerical variable with every other numerical variable.

if len(numeric_cols) >= 2:
    sns.pairplot(df_clean[numeric_cols].dropna())
    plt.show()
else:
    print("At least two numerical columns are needed for a pair plot.")


In [ ]:
# STEP 5C: Example scatter plot
# A scatter plot lets us visually inspect the relationship between two numerical variables.

if len(numeric_cols) >= 2:
    x_col = numeric_cols[0]
    y_col = numeric_cols[1]

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df_clean, x=x_col, y=y_col)
    plt.title(f"{x_col} vs {y_col}")
    plt.tight_layout()
    plt.show()

    print(f"Scatter plot shown for: {x_col} vs {y_col}")


In [ ]:
# STEP 5D: Automatically find the strongest numerical correlations

if len(numeric_cols) >= 2:
    corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    strongest = corr_pairs.stack().sort_values(key=np.abs, ascending=False)

    print("Strongest numerical correlations:")
    display(strongest.head(10).round(3))


# Step 6 — Insights Extraction & Final Report

The most important part of EDA is **not just creating charts**. You must explain what the charts mean.

For each important result, use this pattern:

**Observation → Evidence → Meaning**

Example:
- **Observation:** One group has a higher average value.
- **Evidence:** The group comparison shows a clear difference in the box plot.
- **Meaning:** The variable may be an important factor worth investigating further.

## Final report structure
1. Project Objective
2. Dataset Description
3. Data Cleaning
4. Statistical Summary
5. Univariate Analysis
6. Bivariate & Multivariate Analysis
7. Key Findings
8. Key Influencing Factors
9. Conclusion
10. Future Work

Use `report_template.md` in this project folder to write your final report.


In [ ]:
# STEP 6: Helpful tables for writing your report

print("===== FINAL DATASET SUMMARY =====")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])
print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

print("\n===== TOP CATEGORY VALUES =====")
for col in categorical_cols:
    if df_clean[col].nunique() <= 20:
        print(f"\n{col}")
        display(df_clean[col].value_counts().head(10))

if len(numeric_cols) >= 2:
    print("\n===== STRONGEST CORRELATIONS =====")
    display(strongest.head(10).round(3))

print("\nEDA completed successfully!")


# Submission Checklist

Before submitting, make sure you have:

- [ ] `EDA_Project.ipynb`
- [ ] Statistical summaries
- [ ] Missing-value and duplicate analysis
- [ ] Histograms
- [ ] Box plots
- [ ] Categorical bar charts
- [ ] Correlation heatmap
- [ ] Scatter plot
- [ ] Pair plot
- [ ] Key findings written in `report_template.md`
- [ ] Project uploaded to GitHub
- [ ] GitHub repository link ready for submission
